In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from cafo_iowa.db.session import get_engine
from cafo_iowa.utils.utils import generate_random_points
import datetime

import cafo_iowa.db.models as m
from cafo_iowa.utils.visualize import simple_map

In [ ]:
# Load permits without parcels
permits_no_parcels = gpd.read_postgis(
    """
    SELECT p.* 
    FROM processed.permits p
    LEFT JOIN processed.permit_parcels pp ON p.id = pp.permit_id
    WHERE pp.id IS null
    """,
    get_engine(),
    geom_col="geometry",
)

print(f"Loaded {permits_no_parcels.shape[0]} permits without parcels")

# load barns with no parcels
barns_no_parcels = gpd.read_postgis(
    f"""
    SELECT * 
    FROM processed.barns
    WHERE barn_cluster_id NOT IN (
        SELECT distinct barn_cluster_id 
        FROM processed.barnclusterparcels
    )
    """,
    get_engine(),
    geom_col="geometry",
)

print(f"Loaded {barns_no_parcels.shape[0]} barns without parcels")

# load parcels
parcels = gpd.read_postgis(
    "SELECT * FROM processed.parcels", get_engine(), geom_col="geometry"
)

# get county iowa data
counties = gpd.read_postgis(
    "SELECT * from processed.census_tracts", get_engine(), geom_col="geometry"
)

In [ ]:
# Permits with no parcels
simple_map("EPSG:4326", permits_no_parcels, barns_no_parcels, parcels, counties)

# all but two of the permits with no parcels come from one county (), presumably because ReGrid does not have parcel data for that county

In [ ]:
# Barns with no parcels
simple_map("EPSG:4326", barns_no_parcels, parcels, counties)

In [ ]:
# save permit_no_parcels and barns_no_parcels to csv, with lat and lon
date_today = datetime.datetime.now().strftime("%Y-%m-%d")
permits_no_parcels = permits_no_parcels.to_crs(epsg=4326)
permits_no_parcels["lat"] = permits_no_parcels.geometry.y
permits_no_parcels["lon"] = permits_no_parcels.geometry.x
permits_no_parcels[["id", "lat", "lon"]].to_csv(
    f"data/temp/{date_today}_permits_with_no_parcel.csv", index=False
)

# barns_no_parcels
barns_no_parcels = barns_no_parcels.to_crs(epsg=4326)
barns_no_parcels["lat"] = barns_no_parcels.geometry.centroid.y
barns_no_parcels["lon"] = barns_no_parcels.geometry.centroid.x
barns_no_parcels[["id", "lat", "lon"]].to_csv(
    f"data/temp/{date_today}_barns_with_no_parcel.csv", index=False
)

In [ ]:
# select all barns and permits that are not in page county
barns_no_parcels_not_page = barns_no_parcels[
    ~barns_no_parcels.geometry.within(
        counties[counties.name == "Page"].geometry.values[0]
    )
]
permits_no_parcels_not_page = permits_no_parcels[
    ~permits_no_parcels.geometry.within(
        counties[counties.name == "Page"].geometry.values[0]
    )
]

In [ ]:
# set parameters
num_points = 20
radius = 500

In [ ]:
new_locations = []
for _, row in permits_no_parcels_not_page.iterrows():
    new_locations.extend(
        generate_random_points(row.id, row.geometry, num_points, radius)
    )

# Create new GeoDataFrame
new_locations_gdf = gpd.GeoDataFrame(new_locations, crs=permits_no_parcels_not_page.crs)

# Convert to lat/lon (EPSG:4326) for reGrid
new_locations_gdf = new_locations_gdf.to_crs(epsg=4326)

# Save as CSV with permit_id
new_locations_df = pd.DataFrame(
    {
        "permit_id": new_locations_gdf["id"],
        "latitude": new_locations_gdf.geometry.y,
        "longitude": new_locations_gdf.geometry.x,
    }
)

date_today = datetime.datetime.now().strftime("%Y-%m-%d")

new_locations_df.to_csv(
    f"data/temp/{date_today}_permits_with_no_parcel_{radius}m_jitter.csv", index=False
)

In [ ]:
num_points = 10
new_locations = []
barns_no_parcels_not_page["geometry"] = barns_no_parcels_not_page.centroid
for _, row in barns_no_parcels_not_page.iterrows():
    # find centroid of barn

    new_locations.extend(
        generate_random_points(row.id, row.geometry, num_points, radius)
    )

# Create new GeoDataFrame
new_locations_gdf = gpd.GeoDataFrame(new_locations, crs=barns_no_parcels_not_page.crs)

# Convert to lat/lon (EPSG:4326) for reGrid
new_locations_gdf = new_locations_gdf.to_crs(epsg=4326)

# Save as CSV with permit_id
new_locations_df = pd.DataFrame(
    {
        "permit_id": new_locations_gdf["id"],
        "latitude": new_locations_gdf.geometry.y,
        "longitude": new_locations_gdf.geometry.x,
    }
)

date_today = datetime.datetime.now().strftime("%Y-%m-%d")

new_locations_df.to_csv(
    f"data/temp/{date_today}_barns_with_no_parcel_{radius}m_jitter.csv", index=False
)